# Notebook 2 – Node Feature Engineering
-----------------------------------------------------------
### Purpose

Generate semantic node features for the Metro Manila road network.

### Outputs:

- road_nodes_with_features.geojson
- road_node_features.csv

-----------------------------------------------------------

# Section 1 - Set Up

Imports

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import osmnx as ox
from pathlib import Path

Configurations

In [ ]:
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "osm"

GRAPHS_DIR = DATA_DIR / "graphs"
GEOJSON_DIR = DATA_DIR / "geojson"

CONFIG_DIR = PROJECT_ROOT / "config"

OUTPUT_DIR = PROJECT_ROOT / "data" / "features"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


Feature list

In [ ]:
FEATURE_COLUMNS = [
    "commercial_hubs",
    "service_hubs",
    "financial_service",
    "civic_facilities",
    "educational_facilities",
    "medical_facilities",
    "driver_utility",
    "religious",
    "residential",
]

Each node will have a radius that detects the number of POI's in said radius

In [ ]:
NODE_RADIUS = 100  # meters

# Helper Functions

### POI Functions

Categorizes the POI's into their chosen feature category

In [ ]:
def categorize_pois(gdf, osm_key, feature_map):
    """
    Assign a semantic feature category to each OSM feature.

    Parameters
    ----------
    gdf : GeoDataFrame
        OSM dataset (amenities, buildings, or shops).

    osm_key : str
        OSM tag column ("amenity", "building", or "shop").

    feature_map : DataFrame
        POI feature mapping table.

    Returns
    -------
    GeoDataFrame
        Input GeoDataFrame with an added
        'Feature Category' column.
    """

    gdf = gdf.copy()

    # Build lookup dictionary
    lookup = (
        feature_map[
            feature_map["OSM_Key"] == osm_key
        ]
        .set_index("OSM_Value")["Feature Category"]
        .to_dict()
    )

    # Assign categories
    gdf["feature_category"] = gdf[osm_key].map(lookup)

    return gdf

Merges all the POI datasets into one GeoDataFrame

In [ ]:
def merge_poi_datasets(*datasets):
    """
    Merge multiple categorized POI datasets into a single GeoDataFrame.

    Parameters
    ----------
    *datasets : GeoDataFrame
        Any number of categorized POI datasets.

    Returns
    -------
    GeoDataFrame
        Combined POI dataset.
    """

    columns = [
        "element",
        "id",
        "name",
        "feature_category",
        "geometry",
    ]

    cleaned = []

    for gdf in datasets:

        existing = [
            col for col in columns
            if col in gdf.columns
        ]

        cleaned.append(gdf[existing])

    merged = pd.concat(
        cleaned,
        ignore_index=True,
    )

    return gpd.GeoDataFrame(
        merged,
        geometry="geometry",
        crs=datasets[0].crs,
    )

Convert Polygon POI's to centriod points while leaving point geometries unchanged.

In [ ]:
def represent_pois_as_points(pois_gdf):

    pois = pois_gdf.copy()

    polygon_mask = pois.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )

    converted = polygon_mask.sum()

    pois.loc[polygon_mask, "geometry"] = (
        pois.loc[polygon_mask]
        .geometry
        .centroid
    )

    print(f"Converted {converted:,} polygon POIs to centroids.")

    return pois

### Spatial Functions

Make buffers for each road network node

In [ ]:
def buffer_nodes(nodes_gdf, radius):
    """
    Create circular buffers around each road network node.

    Parameters
    ----------
    nodes_gdf : GeoDataFrame
        Road network nodes.

    radius : float
        Buffer radius in meters.

    Returns
    -------
    GeoDataFrame
        Buffered node geometries.
    """

    buffers = nodes_gdf.copy()

    buffers["geometry"] = buffers.geometry.buffer(radius)

    return buffers

Spatially join the POI's to the node buffers

In [ ]:
def spatially_join_pois(node_buffers, pois_gdf):
    """
    Spatially join POIs to node buffers.

    Parameters
    ----------
    node_buffers : GeoDataFrame
        Buffered road network nodes.

    pois_gdf : GeoDataFrame
        Categorized POIs.

    Returns
    -------
    GeoDataFrame
        Spatial join result.
    """

    print("Performing spatial join...")

    joined = gpd.sjoin(
        pois_gdf,
        node_buffers,
        how="inner",
        predicate="within",
    )
    
    print("Spatial join complete.")

    return joined

### Feature Engineering Functions

Rename each feature category using snake case

In [ ]:
def rename_feature_columns(nodes_gdf):

    rename_dict = {
        "Commercial Hubs": "commercial_hubs",
        "Service Hubs": "service_hubs",
        "Financial Service": "financial_service",
        "Civic Facilities": "civic_facilities",
        "Educational Facilities": "educational_facilities",
        "Medical Facilities": "medical_facilities",
        "Driver Utility": "driver_utility",
        "Residential": "residential",
        "Religious": "religious",
    }

    return nodes_gdf.rename(columns=rename_dict)

Aggregate for each node feature in the radius

In [ ]:
def aggregate_node_features(joined_pois):
    """
    Count POIs of each feature category around every road node.

    Parameters
    ----------
    joined_pois : GeoDataFrame

    Returns
    -------
    DataFrame
        One row per road node with POI category counts.
    """

    node_features = (
        joined_pois
        .groupby(["index_right", "feature_category"])
        .size()
        .unstack(fill_value=0)
    )

    return node_features

Attach the POI features onto the road nodes

In [ ]:
def attach_node_features(nodes_gdf, node_features):
    """
    Merge aggregated POI features onto road nodes.

    Parameters
    ----------
    nodes_gdf : GeoDataFrame
        Road network nodes.

    node_features : DataFrame
        Aggregated POI counts per node.

    Returns
    -------
    GeoDataFrame
        Road nodes with attached feature counts.
    """

    nodes = nodes_gdf.copy()

    nodes = nodes.join(
        node_features,
        how="left",
    )

    feature_columns = node_features.columns

    nodes[feature_columns] = (
        nodes[feature_columns]
        .fillna(0)
        .astype(int)
    )

    return nodes

Keep only selected columns

In [ ]:
def select_node_columns(nodes_gdf):
    columns = [
    "osmid",
    "street_count",
    *FEATURE_COLUMNS,
    "geometry",
    ]

    existing = [
        col
        for col in columns
        if col in nodes_gdf.columns
    ]

    return nodes_gdf[existing]

### Export Functions

Export node features to GeoJSON and CSV

In [ ]:
def export_node_features(nodes_gdf, output_dir):
    
    geojson_path = output_dir / "road_nodes_with_features.geojson"
    csv_path = output_dir / "road_node_features.csv"

    # Export GeoJSON
    nodes_gdf.to_file(
        geojson_path,
        driver="GeoJSON",
    )

    # Convert to a regular DataFrame for CSV export
    csv_df = pd.DataFrame(nodes_gdf.copy())
    csv_df["geometry"] = nodes_gdf.geometry.to_wkt()

    csv_df.to_csv(
        csv_path,
        index=False,
    )

    print(f"Saved: {geojson_path.name}")
    print(f"Saved: {csv_path.name}")

# Section 2 - Load the Processed Datasets

Load the road nodes

In [ ]:
nodes_gdf = gpd.read_file(
    GEOJSON_DIR / "road_nodes.geojson"
)

print(f"Road nodes : {len(nodes_gdf):,}")

Load the POI's (amenities, buildings, and shops)

In [ ]:
amenities = gpd.read_file(
    GEOJSON_DIR / "amenities.geojson"
)

buildings = gpd.read_file(
    GEOJSON_DIR / "buildings.geojson"
)

shops = gpd.read_file(
    GEOJSON_DIR / "shops.geojson"
)

Show all the data loaded

In [ ]:
print("\n========== DATA LOADED ==========")

print(f"Road Nodes : {len(nodes_gdf):,}")

print(f"Amenities : {len(amenities):,}")
print(f"Buildings : {len(buildings):,}")
print(f"Shops     : {len(shops):,}")

print(f"\nCRS : {nodes_gdf.crs}")

# Section 3 - POI Feature Mapping

Load feature map

In [ ]:
CONFIG_DIR = PROJECT_ROOT / "config"
FEATURE_MAP_PATH = CONFIG_DIR / "poi_feature_map.xlsx"

feature_map = pd.read_excel(FEATURE_MAP_PATH)

# Clean text fields
feature_map["Feature Category"] = (
    feature_map["Feature Category"]
    .str.strip()
)

print(f"Loaded {len(feature_map):,} feature mappings.")

Preview

In [ ]:
feature_map.head()

Dataset Info

In [ ]:
feature_map.info()

Check for duplicates or missing values

In [ ]:
duplicates = feature_map.duplicated(
    subset=["OSM_Key", "OSM_Value"],
    keep=False,
)
missing = feature_map["Feature Category"].isna().sum()

print(f"Duplicate mappings: {duplicates.sum()}")
print(f"Missing categories: {missing}")

# Stop program if fault detected
if duplicates.any():
    raise ValueError("Duplicate OSM mappings found.")

if missing > 0:
    raise ValueError("Some OSM features have no assigned category.")

Show feature categories

In [ ]:
print("Feature Categories\n")

for category in sorted(feature_map["Feature Category"].unique()):
    print(f"• {category}")

# Section 4 - Categorize the POI's

Categorize each dataset (amenities, buildings, and shops)

In [ ]:
amenities = categorize_pois(
    amenities,
    "amenity",
    feature_map,
)

buildings = categorize_pois(
    buildings,
    "building",
    feature_map,
)

shops = categorize_pois(
    shops,
    "shop",
    feature_map,
)

Verify results

In [ ]:
print("Amenities")
print(amenities["feature_category"].value_counts(dropna=False))

print("\nBuildings")
print(buildings["feature_category"].value_counts(dropna=False))

print("\nShops")
print(shops["feature_category"].value_counts(dropna=False))

Check for missing mapping

In [ ]:
print("Unmapped POIs")

print(f"Amenities : {amenities['feature_category'].isna().sum():,}")
print(f"Buildings : {buildings['feature_category'].isna().sum():,}")
print(f"Shops     : {shops['feature_category'].isna().sum():,}")

# Stop program if fault detected
if amenities["feature_category"].isna().any():
    raise ValueError("Amenities contain unmapped features.")

if buildings["feature_category"].isna().any():
    raise ValueError("Buildings contain unmapped features.")

if shops["feature_category"].isna().any():
    raise ValueError("Shops contain unmapped features.")

# Section 5 - Merge All Categorized POI's

Merge all the POI dataset into one

In [ ]:
all_pois = merge_poi_datasets(
    amenities,
    buildings,
    shops,
)

print(f"Total POIs: {len(all_pois):,}")

print("\nFeature Categories\n")

print(all_pois["feature_category"].value_counts())

Verify results

In [ ]:
expected = (
    len(amenities)
    + len(buildings)
    + len(shops)
)

print(f"Expected POIs : {expected:,}")
print(f"Merged POIs   : {len(all_pois):,}")

assert len(all_pois) == expected

Convert POI's with polygons as centroid points for easier detection and consistency.

In [ ]:
all_pois = represent_pois_as_points(all_pois)

In [ ]:
all_pois = merge_poi_datasets(
    amenities,
    buildings,
    shops,
)

all_pois = represent_pois_as_points(all_pois)

print(f"Total POIs: {len(all_pois):,}")

print("\nFeature Categories\n")

print(all_pois["feature_category"].value_counts())

In [ ]:
print(all_pois.geometry.geom_type.value_counts())

# Section 6 - Generate Node Features

Create a buffer for each node

In [ ]:
node_buffers = buffer_nodes(
    nodes_gdf,
    radius=NODE_RADIUS,
)

print(f"Created {len(node_buffers):,} node buffers.")

Use spatial join on the buffers and the POI's

In [ ]:
joined_pois = spatially_join_pois(
    node_buffers,
    all_pois,
)

print(f"Joined records: {len(joined_pois):,}")

Keep necessary columns

In [ ]:
joined_pois = joined_pois[
    [
        "index_right",
        "feature_category",
        "geometry",
    ]
]


Aggregate each node feature

In [ ]:
node_features = aggregate_node_features(joined_pois)

print(node_features.shape)
node_features.head()

Attach the node_features table to the nodes_gdf table

In [ ]:
road_nodes_with_features = attach_node_features(
    nodes_gdf,
    node_features,
)

print(road_nodes_with_features.head())

In [ ]:
road_nodes_with_features.iloc[:, -9:].describe()

Clean Node Dataset

In [ ]:
road_nodes_with_features = rename_feature_columns(
    road_nodes_with_features
)

road_nodes_with_features = select_node_columns(
    road_nodes_with_features
)

print(road_nodes_with_features.columns)

Validation check

In [ ]:
nodes_without_pois = (
    road_nodes_with_features[FEATURE_COLUMNS]
    .sum(axis=1)
    .eq(0)
    .sum()
)

print(f"Nodes without nearby POIs: {nodes_without_pois:,}")
print(f"Percentage: {nodes_without_pois / len(road_nodes_with_features) * 100:.2f}%")

In [ ]:
print(road_nodes_with_features.isna().sum())

# Section - Export

In [ ]:
export_node_features(
    road_nodes_with_features,
    OUTPUT_DIR,
)

In [ ]:
print("\n========== NODE FEATURE SUMMARY ==========")

print(f"Road Nodes : {len(road_nodes_with_features):,}")
print(f"Feature Columns : {len(FEATURE_COLUMNS)}")
print(f"Radius : {NODE_RADIUS} meters")

print(f"Nodes with POIs : {len(road_nodes_with_features) - nodes_without_pois:,}")
print(f"Nodes without POIs : {nodes_without_pois:,}")
print(
    f"Coverage : {(1 - nodes_without_pois / len(road_nodes_with_features)) * 100:.2f}%"
)

print(f"\nOutput")
print(OUTPUT_DIR)